<a href="https://colab.research.google.com/github/uvindu827/J26-DS-350/blob/Indhi_Component2_CHL_Forecast/03_CNN_LSTM_CHL_Forecasting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

BASE_DIR = Path(
    "/content/drive/MyDrive/Indhi_Marine_Research_Data"
)

CNN_DIR = (
    BASE_DIR /
    "06_models" /
    "03_CNN_LSTM"
)

EVALUATION_DIR = CNN_DIR / "evaluation"

PREDICTION_DIR = CNN_DIR / "predictions"

CNN_DIR.mkdir(
    parents=True,
    exist_ok=True
)

EVALUATION_DIR.mkdir(
    parents=True,
    exist_ok=True
)

PREDICTION_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("=" * 70)
print("STEP 31.9P-14 — CNN-LSTM PROJECT SETUP")
print("=" * 70)

print("\nCNN-LSTM directory:")
print(CNN_DIR)

print("\nEvaluation directory:")
print(EVALUATION_DIR)

print("\nPrediction directory:")
print(PREDICTION_DIR)

print("\n✓ CNN-LSTM directories created.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
STEP 31.9P-14 — CNN-LSTM PROJECT SETUP

CNN-LSTM directory:
/content/drive/MyDrive/Indhi_Marine_Research_Data/06_models/03_CNN_LSTM

Evaluation directory:
/content/drive/MyDrive/Indhi_Marine_Research_Data/06_models/03_CNN_LSTM/evaluation

Prediction directory:
/content/drive/MyDrive/Indhi_Marine_Research_Data/06_models/03_CNN_LSTM/predictions

✓ CNN-LSTM directories created.


In [1]:
# ======================================================================
# STEP 31.9P-14A — CNN-LSTM GPU VERIFICATION
# ======================================================================

import tensorflow as tf
import sys
import platform

print("=" * 70)
print("STEP 31.9P-14A — CNN-LSTM GPU VERIFICATION")
print("=" * 70)

print("\nPython version:")
print(sys.version)

print("\nTensorFlow version:")
print(tf.__version__)

print("\nPlatform:")
print(platform.platform())

print("\nGPU devices:")
gpus = tf.config.list_physical_devices("GPU")

if gpus:
    for gpu in gpus:
        print(gpu)

    print("\n✓ GPU detected.")
    print("✓ CNN-LSTM can use GPU acceleration.")

else:
    print("No GPU detected.")
    print("⚠️ Do NOT start training yet.")

STEP 31.9P-14A — CNN-LSTM GPU VERIFICATION

Python version:
3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]

TensorFlow version:
2.21.0

Platform:
Linux-6.6.122+-x86_64-with-glibc2.39

GPU devices:
PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')

✓ GPU detected.
✓ CNN-LSTM can use GPU acceleration.


In [2]:
# ======================================================================
# STEP 31.9P-14A — CNN-LSTM ENVIRONMENT + GPU VERIFICATION
# ======================================================================

import sys
import platform
import os

print("=" * 70)
print("STEP 31.9P-14A — CNN-LSTM ENVIRONMENT + GPU VERIFICATION")
print("=" * 70)

# ----------------------------------------------------------------------
# PYTHON / PLATFORM
# ----------------------------------------------------------------------

print("\nPython version:")
print(sys.version)

print("\nPlatform:")
print(platform.platform())

print("\nCUDA_VISIBLE_DEVICES:")
print(os.environ.get("CUDA_VISIBLE_DEVICES"))

# ----------------------------------------------------------------------
# TENSORFLOW VERIFICATION
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("TENSORFLOW VERIFICATION")
print("=" * 70)

try:
    import tensorflow as tf

    print("TensorFlow version:", tf.__version__)

    gpus = tf.config.list_physical_devices("GPU")

    print("Number of GPUs detected:", len(gpus))

    if gpus:
        print("\nGPU devices:")
        for gpu in gpus:
            print(gpu)

        print("\n✓ TensorFlow GPU detected.")
    else:
        print("\n⚠ No GPU detected by TensorFlow.")

except Exception as e:
    print("\nTensorFlow GPU verification failed:")
    print(e)

# ----------------------------------------------------------------------
# CUDA BUILD INFORMATION
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("TENSORFLOW BUILD INFORMATION")
print("=" * 70)

try:
    print("Built with CUDA:",
          tf.test.is_built_with_cuda())

    print("GPU available:",
          len(tf.config.list_physical_devices("GPU")) > 0)

except Exception as e:
    print("Could not retrieve CUDA build information:")
    print(e)

# ----------------------------------------------------------------------
# SIMPLE GPU COMPUTATION TEST
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("GPU COMPUTATION TEST")
print("=" * 70)

try:
    if gpus:
        with tf.device("/GPU:0"):
            test_a = tf.random.normal((2000, 2000))
            test_b = tf.random.normal((2000, 2000))
            test_c = tf.matmul(test_a, test_b)

        print("GPU matrix multiplication completed.")
        print("Result shape:", test_c.shape)

        # Force computation completion
        _ = test_c.numpy()

        print("✓ GPU computation test successful.")

    else:
        print("GPU computation test skipped because no GPU was detected.")

except Exception as e:
    print("GPU computation test failed:")
    print(e)

print("\n" + "=" * 70)
print("STEP 31.9P-14A COMPLETE")
print("=" * 70)

STEP 31.9P-14A — CNN-LSTM ENVIRONMENT + GPU VERIFICATION

Python version:
3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]

Platform:
Linux-6.6.122+-x86_64-with-glibc2.39

CUDA_VISIBLE_DEVICES:
None

TENSORFLOW VERIFICATION
TensorFlow version: 2.21.0
Number of GPUs detected: 1

GPU devices:
PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')

✓ TensorFlow GPU detected.

TENSORFLOW BUILD INFORMATION
Built with CUDA: True
GPU available: True

GPU COMPUTATION TEST
GPU matrix multiplication completed.
Result shape: (2000, 2000)
✓ GPU computation test successful.

STEP 31.9P-14A COMPLETE


In [3]:
# ======================================================================
# STEP 31.9P-14B — CNN-LSTM DATA LOADING & VERIFICATION
# ======================================================================

import os
import numpy as np
import pandas as pd
from pathlib import Path

print("=" * 70)
print("STEP 31.9P-14B — CNN-LSTM DATA LOADING & VERIFICATION")
print("=" * 70)

# ----------------------------------------------------------------------
# PROJECT DIRECTORIES
# ----------------------------------------------------------------------

from google.colab import drive

drive.mount("/content/drive")

BASE_DIR = Path(
    "/content/drive/MyDrive/Indhi_Marine_Research_Data"
)

PREPROCESS_DIR = (
    BASE_DIR /
    "05_model_preprocessing"
)

MODEL_DIR = (
    BASE_DIR /
    "06_models" /
    "03_CNN_LSTM"
)

EVALUATION_DIR = MODEL_DIR / "evaluation"

MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

EVALUATION_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("\nProject directory:")
print(BASE_DIR)

print("\nPreprocessing directory:")
print(PREPROCESS_DIR)

print("\nCNN-LSTM model directory:")
print(MODEL_DIR)

print("\nEvaluation directory:")
print(EVALUATION_DIR)

# ----------------------------------------------------------------------
# FILE PATHS
# ----------------------------------------------------------------------

TRAIN_FILE = (
    PREPROCESS_DIR /
    "final_model_ready_train.csv"
)

VALIDATION_FILE = (
    PREPROCESS_DIR /
    "final_model_ready_validation.csv"
)

TEST_FILE = (
    PREPROCESS_DIR /
    "final_model_ready_test.csv"
)

FEATURE_FILE = (
    PREPROCESS_DIR /
    "final_model_feature_columns.csv"
)

print("\n" + "=" * 70)
print("REQUIRED FILE CHECK")
print("=" * 70)

required_files = {
    "Training dataset": TRAIN_FILE,
    "Validation dataset": VALIDATION_FILE,
    "Test dataset": TEST_FILE,
    "Feature list": FEATURE_FILE
}

for name, path in required_files.items():
    print(f"\n{name}:")
    print(path)
    print("Exists:", path.exists())

    assert path.exists(), f"Missing required file: {path}"

print("\n✓ All required files found.")

# ----------------------------------------------------------------------
# LOAD DATA
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("LOADING MODEL-READY DATA")
print("=" * 70)

print("\nLoading training data...")
train_df = pd.read_csv(TRAIN_FILE)

print("Loading validation data...")
validation_df = pd.read_csv(VALIDATION_FILE)

print("Loading test data...")
test_df = pd.read_csv(TEST_FILE)

print("Loading final feature list...")
feature_df = pd.read_csv(FEATURE_FILE)

print("\n" + "=" * 70)
print("DATASET SHAPES")
print("=" * 70)

print("\nTraining:")
print(train_df.shape)

print("\nValidation:")
print(validation_df.shape)

print("\nTest:")
print(test_df.shape)

# ----------------------------------------------------------------------
# FINAL FEATURE LIST
# ----------------------------------------------------------------------

FEATURES = (
    feature_df["feature"]
    .dropna()
    .astype(str)
    .tolist()
)

FEATURES = list(dict.fromkeys(FEATURES))

print("\n" + "=" * 70)
print("FINAL FEATURE CONFIGURATION")
print("=" * 70)

print("\nNumber of selected features:")
print(len(FEATURES))

assert len(FEATURES) == 69

print("\n✓ Exactly 69 final predictors loaded.")

# ----------------------------------------------------------------------
# TARGET CONFIGURATION
# ----------------------------------------------------------------------

TARGET_LOG = "chlorophyll_a_log1p"
TARGET_ORIGINAL = "chlorophyll_a"

print("\n" + "=" * 70)
print("TARGET CONFIGURATION")
print("=" * 70)

print("\nLog target:")
print(TARGET_LOG)

print("\nOriginal target:")
print(TARGET_ORIGINAL)

# ----------------------------------------------------------------------
# FEATURE EXISTENCE CHECK
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("FEATURE EXISTENCE CHECK")
print("=" * 70)

for dataset_name, df in [
    ("Training", train_df),
    ("Validation", validation_df),
    ("Test", test_df)
]:

    missing_features = [
        feature for feature in FEATURES
        if feature not in df.columns
    ]

    print(f"\n{dataset_name} missing features:")
    print(len(missing_features))

    assert len(missing_features) == 0

    print(f"✓ All 69 features available in {dataset_name}.")

# ----------------------------------------------------------------------
# TARGET EXISTENCE CHECK
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("TARGET EXISTENCE CHECK")
print("=" * 70)

for dataset_name, df in [
    ("Training", train_df),
    ("Validation", validation_df),
    ("Test", test_df)
]:

    assert TARGET_LOG in df.columns
    assert TARGET_ORIGINAL in df.columns

    print(f"\n✓ {dataset_name}: both target representations available.")

# ----------------------------------------------------------------------
# METADATA CHECK
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("METADATA CHECK")
print("=" * 70)

METADATA_COLUMNS = [
    "date",
    "latitude",
    "longitude",
    "data_split"
]

for dataset_name, df in [
    ("Training", train_df),
    ("Validation", validation_df),
    ("Test", test_df)
]:

    missing_metadata = [
        col for col in METADATA_COLUMNS
        if col not in df.columns
    ]

    print(f"\n{dataset_name} missing metadata columns:")
    print(missing_metadata)

    assert len(missing_metadata) == 0

print("\n✓ Required metadata columns available.")

# ----------------------------------------------------------------------
# MISSING VALUE CHECK
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("MISSING VALUE CHECK")
print("=" * 70)

for dataset_name, df in [
    ("Training", train_df),
    ("Validation", validation_df),
    ("Test", test_df)
]:

    predictor_missing = df[FEATURES].isna().sum().sum()
    log_target_missing = df[TARGET_LOG].isna().sum()
    original_target_missing = df[TARGET_ORIGINAL].isna().sum()

    print(f"\n{dataset_name}:")
    print("Predictor missing values:", predictor_missing)
    print("Log target missing values:", log_target_missing)
    print("Original target missing values:", original_target_missing)

    assert predictor_missing == 0
    assert log_target_missing == 0
    assert original_target_missing == 0

print("\n✓ No missing predictor or target values.")

# ----------------------------------------------------------------------
# INFINITE VALUE CHECK
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("INFINITE VALUE CHECK")
print("=" * 70)

for dataset_name, df in [
    ("Training", train_df),
    ("Validation", validation_df),
    ("Test", test_df)
]:

    infinite_count = np.isinf(
        df[FEATURES].to_numpy(dtype=np.float64)
    ).sum()

    print(f"\n{dataset_name} infinite predictor values:")
    print(infinite_count)

    assert infinite_count == 0

print("\n✓ No infinite predictor values.")

# ----------------------------------------------------------------------
# TARGET LEAKAGE CHECK
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("TARGET LEAKAGE CHECK")
print("=" * 70)

print("\nLog target included as predictor:")
print(TARGET_LOG in FEATURES)

print("Original CHL included as predictor:")
print(TARGET_ORIGINAL in FEATURES)

assert TARGET_LOG not in FEATURES
assert TARGET_ORIGINAL not in FEATURES

print("\n✓ No target leakage detected.")

# ----------------------------------------------------------------------
# DATE CHECK
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("DATE VERIFICATION")
print("=" * 70)

for dataset_name, df in [
    ("Training", train_df),
    ("Validation", validation_df),
    ("Test", test_df)
]:

    date_series = pd.to_datetime(df["date"])

    print(f"\n{dataset_name}:")
    print("Minimum date:", date_series.min())
    print("Maximum date:", date_series.max())
    print("Chronologically ordered:",
          date_series.is_monotonic_increasing)

    assert date_series.is_monotonic_increasing

print("\n✓ All datasets are chronologically ordered.")

# ----------------------------------------------------------------------
# DATA SPLIT OVERLAP CHECK
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("DATA SPLIT OVERLAP CHECK")
print("=" * 70)

train_dates = set(train_df["date"].astype(str))
validation_dates = set(validation_df["date"].astype(str))
test_dates = set(test_df["date"].astype(str))

train_validation_overlap = len(
    train_dates.intersection(validation_dates)
)

validation_test_overlap = len(
    validation_dates.intersection(test_dates)
)

train_test_overlap = len(
    train_dates.intersection(test_dates)
)

print("\nTrain ∩ Validation:")
print(train_validation_overlap)

print("Validation ∩ Test:")
print(validation_test_overlap)

print("Train ∩ Test:")
print(train_test_overlap)

assert train_validation_overlap == 0
assert validation_test_overlap == 0
assert train_test_overlap == 0

print("\n✓ No date overlap between datasets.")

# ----------------------------------------------------------------------
# SPATIAL CHECK
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("SPATIAL INFORMATION CHECK")
print("=" * 70)

for dataset_name, df in [
    ("Training", train_df),
    ("Validation", validation_df),
    ("Test", test_df)
]:

    print(f"\n{dataset_name}:")
    print("Latitude range:",
          df["latitude"].min(),
          "to",
          df["latitude"].max())

    print("Longitude range:",
          df["longitude"].min(),
          "to",
          df["longitude"].max())

print("\n✓ Spatial information preserved.")

# ----------------------------------------------------------------------
# FINAL SUMMARY
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("STEP 31.9P-14B SUMMARY")
print("=" * 70)

print("\nTraining rows:", len(train_df))
print("Validation rows:", len(validation_df))
print("Test rows:", len(test_df))

print("\nSelected predictors:", len(FEATURES))

print("\nLog target:", TARGET_LOG)
print("Original target:", TARGET_ORIGINAL)

print("\n✓ Same 69-feature configuration preserved.")
print("✓ Original and log CHL targets preserved.")
print("✓ No missing predictor values.")
print("✓ No infinite predictor values.")
print("✓ No target leakage.")
print("✓ Chronological ordering preserved.")
print("✓ No date overlap between splits.")
print("✓ Spatial information preserved.")
print("✓ Test data has NOT been used for modelling decisions.")

print("\n" + "=" * 70)
print("STEP 31.9P-14B COMPLETE")
print("=" * 70)

STEP 31.9P-14B — CNN-LSTM DATA LOADING & VERIFICATION
Mounted at /content/drive

Project directory:
/content/drive/MyDrive/Indhi_Marine_Research_Data

Preprocessing directory:
/content/drive/MyDrive/Indhi_Marine_Research_Data/05_model_preprocessing

CNN-LSTM model directory:
/content/drive/MyDrive/Indhi_Marine_Research_Data/06_models/03_CNN_LSTM

Evaluation directory:
/content/drive/MyDrive/Indhi_Marine_Research_Data/06_models/03_CNN_LSTM/evaluation

REQUIRED FILE CHECK

Training dataset:
/content/drive/MyDrive/Indhi_Marine_Research_Data/05_model_preprocessing/final_model_ready_train.csv
Exists: True

Validation dataset:
/content/drive/MyDrive/Indhi_Marine_Research_Data/05_model_preprocessing/final_model_ready_validation.csv
Exists: True

Test dataset:
/content/drive/MyDrive/Indhi_Marine_Research_Data/05_model_preprocessing/final_model_ready_test.csv
Exists: True

Feature list:
/content/drive/MyDrive/Indhi_Marine_Research_Data/05_model_preprocessing/final_model_feature_columns.csv
Exis